In [1]:
import logging

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [2]:
spark = (
    SparkSession.builder 
    .appName("MinIO test")
    .master("local[*]")  
    .config("spark.jars.packages", 
        "org.apache.hadoop:hadoop-aws:3.3.4,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.262") 
    .config("spark.hadoop.fs.s3a.endpoint", "http://127.0.0.1:9050") 
    .config("spark.hadoop.fs.s3a.access.key", "adm") # root name
    .config("spark.hadoop.fs.s3a.secret.key", "secret_pass") # root password
    .config("spark.hadoop.fs.s3a.path.style.access", "true") 
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") 
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") 
    .getOrCreate()
)

In [3]:
df = spark.read.parquet("s3a://raw-data/taxi-raw/*.parquet")

In [4]:
df.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- originating_base_num: string (nullable = true)
 |-- request_datetime: timestamp_ntz (nullable = true)
 |-- on_scene_datetime: timestamp_ntz (nullable = true)
 |-- pickup_datetime: timestamp_ntz (nullable = true)
 |-- dropoff_datetime: timestamp_ntz (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- trip_time: long (nullable = true)
 |-- base_passenger_fare: double (nullable = true)
 |-- tolls: double (nullable = true)
 |-- bcf: double (nullable = true)
 |-- sales_tax: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- tips: double (nullable = true)
 |-- driver_pay: double (nullable = true)
 |-- shared_request_flag: string (nullable = true)
 |-- shared_match_flag: string (nullable = true)
 |-- access_a_

### Сделаем выборку без аномалий

In [5]:
df_filer_one = df.filter(
   (df.trip_miles > 0) & (df.trip_time > 0) & (df.driver_pay > 0) & (df.tips >= 0)
)

### Приводим к желаемому типу

In [6]:
df_filer_one = df_filer_one.withColumn(
    "request_datetime", F.to_timestamp(F.col("request_datetime"), "yyyy-MM-dd HH:mm:ss")
).withColumn(
    "on_scene_datetime", F.to_timestamp(F.col("on_scene_datetime"), "yyyy-MM-dd HH:mm:ss")
).withColumn(
    "pickup_datetime", F.to_timestamp(F.col("pickup_datetime"), "yyyy-MM-dd HH:mm:ss")
).withColumn(
    "dropoff_datetime", F.to_timestamp(F.col("dropoff_datetime"), "yyyy-MM-dd HH:mm:ss")
)

In [7]:
# Добавим еще одну колонку с только с датой поступления запроса
df_filer_one = df_filer_one.withColumn(
    "request_date", F.to_date(F.col("request_datetime"))
)

### Заполним пропуски

In [8]:
df_filer_one = df_filer_one.fillna({
    "originating_base_num":"UNKWN"   
})  

### Общая плата за проезд

In [9]:
df_filer_one = df_filer_one.withColumn(
    "total_price", F.col("base_passenger_fare") + F.col("tolls") + F.col("bcf") + F.col("sales_tax") + F.col("congestion_surcharge") + F.col("airport_fee")
)

In [10]:
df_filer_one.explain()

== Physical Plan ==
*(1) Project [hvfhs_license_num#0, dispatching_base_num#1, coalesce(originating_base_num#2, UNKWN) AS originating_base_num#208, request_datetime#51, on_scene_datetime#77, pickup_datetime#103, dropoff_datetime#129, PULocationID#7, DOLocationID#8, trip_miles#9, trip_time#10L, base_passenger_fare#11, tolls#12, bcf#13, sales_tax#14, congestion_surcharge#15, airport_fee#16, tips#17, driver_pay#18, shared_request_flag#19, shared_match_flag#20, access_a_ride_flag#21, wav_request_flag#22, wav_match_flag#23, ... 3 more fields]
+- *(1) Project [hvfhs_license_num#0, dispatching_base_num#1, originating_base_num#2, gettimestamp(request_datetime#3, yyyy-MM-dd HH:mm:ss, TimestampType, Some(Europe/Moscow), false) AS request_datetime#51, gettimestamp(on_scene_datetime#4, yyyy-MM-dd HH:mm:ss, TimestampType, Some(Europe/Moscow), false) AS on_scene_datetime#77, gettimestamp(pickup_datetime#5, yyyy-MM-dd HH:mm:ss, TimestampType, Some(Europe/Moscow), false) AS pickup_datetime#103, gettim

### Сделаем выбор только определенных призанков

In [11]:
# Что возьмем для сохранения

#  |-- request_datetime: timestamp_ntz (nullable = true)    --- Дата и время, когда пассажир запросил поезду
#  |-- on_scene_datetime: timestamp_ntz (nullable = true)   --- Дата и время прибытия водителя на место посадки
#  |-- pickup_datetime: timestamp_ntz (nullable = true)     --- Дата и время начала поездки
#  |-- dropoff_datetime: timestamp_ntz (nullable = true)    --- Дата и время высадки
#  |-- PULocationID: integer (nullable = true)              --- ID зоны-посадки
#  |-- DOLocationID: integer (nullable = true)              --- ID зоны-высадки
#  |-- trip_miles: double (nullable = true)                 --- Общее растояние поездки (в милях)
#  |-- total_price: double                                  --- Добавим колонку со всей ценой за поездку
#  |-- tips: double (nullable = true)                       --- Общая сумма чаевых, полученных от пассажира.
#  |-- driver_pay: double (nullable = true)                 --- Общая заработная плата водителя (без учета платы за проезд по платным дорогам и чаевых, а также за вычетом комиссионных, дополнительных сборов и налогов).
#  |-- shared_request_flag: string (nullable = true)        --- Согласился ли пассажир на совместную поездку, независимо от того, были ли они подобраны друг другу? (Да/Нет)
#  |-- shared_match_flag: string (nullable = true)          --- Ехал ли пассажир вместе с другим пассажиром, забронировавшим поездку отдельно

In [12]:
df_filer_one = df_filer_one.select(
    F.col("request_datetime"),
    F.col("on_scene_datetime"),
    F.col("pickup_datetime"),
    F.col("dropoff_datetime"),
    F.col("PULocationID"),
    F.col("DOLocationID"),
    F.col("trip_miles"),
    F.col("total_price"),
    F.col("tips"),
    F.col("driver_pay"),
    F.col("shared_request_flag"),
    F.col("shared_match_flag"),
    F.col("request_date")
)

In [13]:
df_filer_one.count()

38907246

In [13]:
df_filer_one.rdd.getNumPartitions()

12

In [14]:
df_sample = df_filer_one.sample(withReplacement=False, fraction=0.05, seed=42)

### Сколько пропусков

In [ ]:
# # === Проверка на пропуски ===
# df_sample.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_sample.columns]).show(vertical=True)

ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "c:\dev\mlRes\mlenv\lib\site-packages\pyspark\errors\exceptions\captured.py", line 179, in deco
    return f(*a, **kw)
  File "c:\dev\mlRes\mlenv\lib\site-packages\py4j\protocol.py", line 326, in get_return_value
    raise Py4JJavaError(
py4j.protocol.Py4JJavaError: <exception str() failed>

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "c:\dev\mlRes\mlenv\lib\site-packages\py4j\clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.10_3.10.3056.0_x64__qbz5n2kfra8p0\lib\socket.py", line 705, in readinto
    return self._sock.recv_into(b)
ConnectionResetError: [WinError 10054] Удаленный хост принудительно разорвал существующее подключение

During handling of the above exception, another exception occurred:

Traceback (most re

Py4JError: py4j does not exist in the JVM

In [ ]:
df_filer_one.show(10)

In [14]:
df_filer_one.show(2, vertical=True)

-RECORD 0----------------------------------
 request_datetime    | 2025-07-01 03:05:56 
 on_scene_datetime   | 2025-07-01 03:08:03 
 pickup_datetime     | 2025-07-01 03:08:44 
 dropoff_datetime    | 2025-07-01 03:28:42 
 PULocationID        | 65                  
 DOLocationID        | 179                 
 trip_miles          | 9.93                
 total_price         | 29.179999999999996  
 tips                | 0.0                 
 driver_pay          | 26.11               
 shared_request_flag | N                   
 shared_match_flag   | N                   
 request_date        | 2025-07-01          
-RECORD 1----------------------------------
 request_datetime    | 2025-07-01 03:09:38 
 on_scene_datetime   | 2025-07-01 03:16:28 
 pickup_datetime     | 2025-07-01 03:17:28 
 dropoff_datetime    | 2025-07-01 03:22:50 
 PULocationID        | 35                  
 DOLocationID        | 35                  
 trip_miles          | 0.792               
 total_price         | 6.99     

### Запись частями

In [15]:
date_group = df_filer_one.groupBy(F.col("request_date")).agg(
    F.count("*").alias("row_count")
)

In [16]:
date_group.show(70)

+------------+---------+
|request_date|row_count|
+------------+---------+
|  2025-07-01|   523478|
|  2025-07-05|   616979|
|  2025-07-03|   621276|
|  2025-07-06|   585734|
|  2025-07-04|   557282|
|  2025-07-02|   579824|
|  2025-07-09|   595422|
|  2025-07-08|   586398|
|  2025-07-07|   539173|
|  2025-07-11|   665118|
|  2025-07-10|   622487|
|  2025-07-12|   736603|
|  2025-07-15|   587726|
|  2025-07-16|   631279|
|  2025-07-13|   683497|
|  2025-07-14|   625469|
|  2025-07-18|   676312|
|  2025-07-21|   568924|
|  2025-07-20|   698011|
|  2025-07-17|   667336|
|  2025-07-22|   555913|
|  2025-07-19|   740562|
|  2025-07-27|   686342|
|  2025-07-23|   592311|
|  2025-07-24|   629895|
|  2025-07-26|   736532|
|  2025-07-25|   706467|
|  2025-08-05|   555809|
|  2025-08-04|   555355|
|  2025-08-02|   705344|
|  2025-08-03|   660980|
|  2025-08-01|   660255|
|  2025-08-06|   583799|
|  2025-08-08|   648175|
|  2025-08-10|   651895|
|  2025-08-07|   610122|
|  2025-08-11|   547320|


In [16]:
dates = date_group.collect()

In [17]:
for item in dates:
    date, row_count = item
    sub_df = df_filer_one.filter(F.col("request_date") == date)
    
    print(str(date))
    sub_df.write.mode("overwrite").parquet(f"s3a://silver-data/date-part/{str(date)}") # указать свой путь для сохранения
    break

2025-07-01


ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "c:\dev\mlRes\mlenv\lib\site-packages\py4j\clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.10_3.10.3056.0_x64__qbz5n2kfra8p0\lib\socket.py", line 705, in readinto
    return self._sock.recv_into(b)
ConnectionResetError: [WinError 10054] Удаленный хост принудительно разорвал существующее подключение

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "c:\dev\mlRes\mlenv\lib\site-packages\py4j\java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "c:\dev\mlRes\mlenv\lib\site-packages\py4j\clientserver.py", line 539, in send_command
    raise Py4JNetworkError(
py4j.protocol.Py4JNetworkError: Error while sending or receiving


Py4JError: An error occurred while calling o124.parquet